# OntologySim - Bronze Ingestion

Bronze ingestion for the DEV generation.

Sources:
- Test Management — CSV
- Test Execution — JSONL
- ConMod Runtime — JSONL
- HCP5 Runtime — Log + Quarantine
- HCP3 Runtime — DLT-like Log + Quarantine
- Trace Export — TSV
- Requirements — ReqIF/XML
- Architecture — ARXML
- Defects / KPM — XLSX

In [0]:
import json
import uuid

from pyspark.sql import functions as F

generation_id = "8573dcf32f2ed486"
profile = "medium"

landing_base = (
    f"/Volumes/ontologysim/landing/source_files/"
    f"{profile}/{generation_id}"
)

manifest_path = f"{landing_base}/manifest.json"

manifest = json.loads(dbutils.fs.head(manifest_path, 100_000))

if manifest["generation_id"] != generation_id:
    raise ValueError("Generation ID does not match")

if manifest["profile"] != profile:
    raise ValueError("Profile does not match")

ingestion_run_id = str(uuid.uuid4())

print(f"Generation:    {generation_id}")
print(f"Profile:       {profile}")
print(f"Ingestion run: {ingestion_run_id}")

Generation:    8573dcf32f2ed486
Profile:       medium
Ingestion run: 18970ef1-7f1d-4012-9d28-dc72174e7ee2


In [0]:
def write_generation(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option(
            "replaceWhere",
            f"_generation_id = '{generation_id}'",
        )
        .saveAsTable(table_name)
    )

## Landing Validation

In [0]:
landing_errors = []

for expected_file in manifest["files"]:
    relative_path = expected_file["path"]
    expected_size = expected_file["size_bytes"]

    full_path = f"{landing_base}/{relative_path}"
    parent_path, file_name = full_path.rsplit("/", 1)

    landed_files = {
        item.name.rstrip("/"): item.size
        for item in dbutils.fs.ls(parent_path)
    }

    if file_name not in landed_files:
        landing_errors.append(
            f"Missing file: {relative_path}"
        )
        continue

    actual_size = landed_files[file_name]

    if actual_size != expected_size:
        landing_errors.append(
            f"Size mismatch for {relative_path}: "
            f"expected {expected_size}, got {actual_size}"
        )

if landing_errors:
    raise ValueError("\n".join(landing_errors))

print(
    f"Landing validation passed: "
    f"{len(manifest['files'])} source files verified"
)

Landing validation passed: 10 source files verified


## Test Management — CSV

In [0]:
test_cases_path = f"{landing_base}/test_management/test_cases.csv"

test_cases_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(test_cases_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("test_management"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

write_generation(
    test_cases_df,
    "ontologysim.bronze.test_cases",
)

expected_rows = manifest["source_counts"]["test_management"]
actual_rows = (
    spark.table("ontologysim.bronze.test_cases")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print(f"Expected rows: {expected_rows}")
print(f"Bronze rows:   {actual_rows}")

if expected_rows != actual_rows:
    raise ValueError("Test Management row count mismatch")

print("Test Management validation passed")

Expected rows: 16080
Bronze rows:   16080
Test Management validation passed


## Test Execution — JSONL

In [0]:
test_execution_path = f"{landing_base}/test_execution/test_executions.jsonl"

test_executions_df = (
    spark.read
    .json(test_execution_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("test_execution"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

write_generation(
    test_executions_df,
    "ontologysim.bronze.test_executions",
)

expected_rows = manifest["source_counts"]["test_execution"]
actual_rows = (
    spark.table("ontologysim.bronze.test_executions")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print(f"Expected rows: {expected_rows}")
print(f"Bronze rows:   {actual_rows}")

if expected_rows != actual_rows:
    raise ValueError("Test Execution row count mismatch")

print("Test Execution validation passed")

Expected rows: 160800
Bronze rows:   160800
Test Execution validation passed


## ConMod Runtime — JSONL

In [0]:
conmod_path = f"{landing_base}/runtime/conmod/conmod_runtime.jsonl"

conmod_bronze_df = (
    spark.read
    .json(conmod_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("conmod_runtime"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

write_generation(
    conmod_bronze_df,
    "ontologysim.bronze.conmod_runtime",
)

expected_rows = manifest["source_counts"]["conmod_logs"]
actual_rows = (
    spark.table("ontologysim.bronze.conmod_runtime")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print(f"Expected rows: {expected_rows}")
print(f"Bronze rows:   {actual_rows}")

if expected_rows != actual_rows:
    raise ValueError("ConMod ingestion count mismatch")

print("ConMod validation passed")

Expected rows: 16497867
Bronze rows:   16497867
ConMod validation passed


## HCP5 Runtime — Log + Quarantine

In [0]:
hcp5_path = f"{landing_base}/runtime/hcp5/hcp5_runtime.log"

hcp5_raw_df = (
    spark.read
    .text(hcp5_path)
    .select(
        F.col("value").alias("_raw_line"),
        F.col("_metadata.file_path").alias("_source_file"),
    )
)

hcp5_corrupted_df = hcp5_raw_df.filter(
    F.col("_raw_line").contains(" CORRUPTED ")
)

hcp5_candidate_df = hcp5_raw_df.filter(
    ~F.col("_raw_line").contains(" CORRUPTED ")
)

print(f"Corrupted lines:  {hcp5_corrupted_df.count()}")
print(f"Candidate lines:  {hcp5_candidate_df.count()}")

Corrupted lines:  22726
Candidate lines:  11406745


In [0]:
hcp5_pattern = (
    r'^(\d{2}\.\d{2}\.\d{4} \d{2}:\d{2}:\d{2}\.\d{3}) '
    r'\[([^\]]*)\] '
    r'ecu=(\S*) '
    r'component=(\S*) '
    r'event=(\S*) '
    r'session=(\S*) '
    r'execution=(\S*) '
    r'sw=(\S*) '
    r'msg="(.*)"$'
)

hcp5_unexpected_rejected_df = (
    hcp5_candidate_df
    .filter(~F.col("_raw_line").rlike(hcp5_pattern))
)

hcp5_valid_raw_df = (
    hcp5_candidate_df
    .filter(F.col("_raw_line").rlike(hcp5_pattern))
)

hcp5_parsed_df = (
    hcp5_valid_raw_df
    .select(
        F.regexp_extract("_raw_line", hcp5_pattern, 1).alias("timestamp"),
        F.regexp_extract("_raw_line", hcp5_pattern, 2).alias("severity"),
        F.regexp_extract("_raw_line", hcp5_pattern, 3).alias("ecu"),
        F.regexp_extract("_raw_line", hcp5_pattern, 4).alias("component"),
        F.regexp_extract("_raw_line", hcp5_pattern, 5).alias("event"),
        F.regexp_extract("_raw_line", hcp5_pattern, 6).alias("session_id"),
        F.regexp_extract("_raw_line", hcp5_pattern, 7).alias("execution_id"),
        F.regexp_extract("_raw_line", hcp5_pattern, 8).alias("software_version"),
        F.regexp_extract("_raw_line", hcp5_pattern, 9).alias("message"),
        "_raw_line",
        "_source_file",
    )
)

print(f"Parsed records:            {hcp5_parsed_df.count()}")
print(f"Unexpected parse failures: {hcp5_unexpected_rejected_df.count()}")

Parsed records:            11406745
Unexpected parse failures: 0


In [0]:
hcp5_bronze_df = (
    hcp5_parsed_df
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("hcp5_runtime"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

hcp5_known_rejected_df = (
    hcp5_corrupted_df
    .withColumn(
        "_reject_reason",
        F.lit("CORRUPTED_SOURCE_RECORD"),
    )
)

hcp5_unexpected_rejected_df = (
    hcp5_unexpected_rejected_df
    .withColumn(
        "_reject_reason",
        F.lit("UNEXPECTED_PARSE_FAILURE"),
    )
)

hcp5_rejected_df = hcp5_known_rejected_df.unionByName(
    hcp5_unexpected_rejected_df
)

hcp5_quarantine_df = (
    hcp5_rejected_df
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("hcp5_runtime"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

write_generation(
    hcp5_bronze_df,
    "ontologysim.bronze.hcp5_runtime",
)

write_generation(
    hcp5_quarantine_df,
    "ontologysim.quarantine.hcp5_runtime",
)

expected_rows = manifest["source_counts"]["hcp5_logs"]

bronze_rows = (
    spark.table("ontologysim.bronze.hcp5_runtime")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

quarantine_rows = (
    spark.table("ontologysim.quarantine.hcp5_runtime")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

accounted_rows = bronze_rows + quarantine_rows

print(f"Expected source rows: {expected_rows}")
print(f"Bronze rows:          {bronze_rows}")
print(f"Quarantine rows:      {quarantine_rows}")
print(f"Accounted rows:       {accounted_rows}")

if accounted_rows != expected_rows:
    raise ValueError("HCP5 ingestion count mismatch")

print("HCP5 validation passed")

Expected source rows: 11429471
Bronze rows:          11406745
Quarantine rows:      22726
Accounted rows:       11429471
HCP5 validation passed


## HCP3 Runtime — DLT-like Log + Quarantine

In [0]:
hcp3_path = f"{landing_base}/runtime/hcp3/hcp3_runtime.dlt"

hcp3_raw_df = (
    spark.read
    .text(hcp3_path)
    .select(
        F.col("value").alias("_raw_line"),
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_parts", F.split(F.col("_raw_line"), r"\|", -1))
)

hcp3_valid_raw_df = hcp3_raw_df.filter(
    F.size("_parts") == 12
)

hcp3_rejected_df = hcp3_raw_df.filter(
    F.size("_parts") != 12
)

print(f"Valid lines:    {hcp3_valid_raw_df.count()}")
print(f"Rejected lines: {hcp3_rejected_df.count()}")

Valid lines:    10643872
Rejected lines: 21344


In [0]:
hcp3_parsed_df = hcp3_valid_raw_df.select(
    F.col("_parts")[0].alias("timestamp"),
    F.col("_parts")[1].alias("ecu_id"),
    F.col("_parts")[2].alias("application_id"),
    F.col("_parts")[3].alias("context_id"),
    F.col("_parts")[4].alias("component_name"),
    F.col("_parts")[5].alias("severity"),
    F.col("_parts")[6].alias("event_type"),
    F.col("_parts")[7].alias("message"),
    F.col("_parts")[8].alias("session_id"),
    F.col("_parts")[9].alias("execution_id"),
    F.col("_parts")[10].alias("software_version"),
    F.col("_parts")[11].alias("thread_id"),
    "_raw_line",
    "_source_file",
)

print(f"Parsed records: {hcp3_parsed_df.count()}")

Parsed records: 10643872


In [0]:
hcp3_bronze_df = (
    hcp3_parsed_df
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("hcp3_runtime"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

hcp3_quarantine_df = (
    hcp3_rejected_df
    .drop("_parts")
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("hcp3_runtime"))
    .withColumn("_reject_reason", F.lit("INVALID_FIELD_COUNT"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

write_generation(
    hcp3_bronze_df,
    "ontologysim.bronze.hcp3_runtime",
)

write_generation(
    hcp3_quarantine_df,
    "ontologysim.quarantine.hcp3_runtime",
)

expected_rows = manifest["source_counts"]["hcp3_logs"]

bronze_rows = (
    spark.table("ontologysim.bronze.hcp3_runtime")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)
quarantine_rows = (
    spark.table("ontologysim.quarantine.hcp3_runtime")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

accounted_rows = bronze_rows + quarantine_rows

print(f"Expected source rows: {expected_rows}")
print(f"Bronze rows:          {bronze_rows}")
print(f"Quarantine rows:      {quarantine_rows}")
print(f"Accounted rows:       {accounted_rows}")

if accounted_rows != expected_rows:
    raise ValueError("HCP3 ingestion count mismatch")

print("HCP3 validation passed")

Expected source rows: 10665216
Bronze rows:          10643872
Quarantine rows:      21344
Accounted rows:       10665216
HCP3 validation passed


## Trace Export — TSV

In [0]:
trace_path = f"{landing_base}/traces/trace_export.tsv"

trace_bronze_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("inferSchema", False)
    .csv(trace_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("traces"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

write_generation(
    trace_bronze_df,
    "ontologysim.bronze.traces",
)

expected_rows = manifest["source_counts"]["traces"]
actual_rows = (
    spark.table("ontologysim.bronze.traces")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print(f"Expected rows: {expected_rows}")
print(f"Bronze rows:   {actual_rows}")

if expected_rows != actual_rows:
    raise ValueError("Trace ingestion count mismatch")

print("Trace validation passed")

Expected rows: 155976
Bronze rows:   155976
Trace validation passed


## Requirements — ReqIF / XML

In [0]:
requirements_path = f"{landing_base}/requirements/requirements.reqif"

requirements_bronze_df = (
    spark.read
    .format("xml")
    .option("rowTag", "SPEC-OBJECT")
    .option("inferSchema", False)
    .load(requirements_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("requirements"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

requirements_bronze_df.printSchema()

write_generation(
    requirements_bronze_df,
    "ontologysim.bronze.requirements",
)

expected_rows = manifest["source_counts"]["requirements"]
actual_rows = (
    spark.table("ontologysim.bronze.requirements")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print(f"Expected rows: {expected_rows}")
print(f"Bronze rows:   {actual_rows}")

if expected_rows != actual_rows:
    raise ValueError("Requirements ingestion count mismatch")

print("Requirements validation passed")

root
 |-- CREATED-AT: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- OWNER-TEAM: string (nullable = true)
 |-- PARENT-REQUIREMENT-ID: string (nullable = true)
 |-- PRIORITY: string (nullable = true)
 |-- RELATED-REQUIREMENT-IDS: string (nullable = true)
 |-- REQUIREMENT-ID: string (nullable = true)
 |-- REQUIREMENT-TYPE: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- TARGET-COMPONENT: string (nullable = true)
 |-- TARGET-ECU: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- UPDATED-AT: string (nullable = true)
 |-- VERIFICATION-METHOD: string (nullable = true)
 |-- VERSION: string (nullable = true)
 |-- _source_file: string (nullable = false)
 |-- _generation_id: string (nullable = false)
 |-- _profile: string (nullable = false)
 |-- _source_family: string (nullable = false)
 |-- _ingestion_run_id: string (nullable = false)
 |-- _ingestion_timestamp: timestamp (nullable = false)

Expected rows: 8040
Bronze rows:   8040
Re

## Architecture — ARXML

In [0]:
architecture_path = f"{landing_base}/architecture/architecture.arxml"

components_df = (
    spark.read
    .format("xml")
    .option("rowTag", "COMPONENT")
    .option("inferSchema", False)
    .load(architecture_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("architecture"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

ecus_df = (
    spark.read
    .format("xml")
    .option("rowTag", "ECU")
    .option("inferSchema", False)
    .load(architecture_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("architecture"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

In [0]:
write_generation(
    components_df,
    "ontologysim.bronze.architecture_components",
)

write_generation(
    ecus_df,
    "ontologysim.bronze.architecture_ecus",
)

expected_components = manifest["source_counts"]["components"]

actual_components = (
    spark.table("ontologysim.bronze.architecture_components")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

actual_ecus = (
    spark.table("ontologysim.bronze.architecture_ecus")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print(f"Expected components: {expected_components}")
print(f"Bronze components:   {actual_components}")
print(f"Bronze ECUs:         {actual_ecus}")

if expected_components != actual_components:
    raise ValueError("Architecture component count mismatch")

print("Architecture validation passed")

Expected components: 1600
Bronze components:   1600
Bronze ECUs:         3
Architecture validation passed


## Defects / KPM — XLSX

In [0]:
defects_path = f"{landing_base}/defects/kpm_defects.xlsx"

defects_raw_df = (
    spark.read
    .format("excel")
    .option("headerRows", 1)
    .option("dataAddress", "KPM")
    .load(defects_path)
)

# Keep source fields as strings in Bronze.
source_columns = defects_raw_df.columns

defects_bronze_df = (
    defects_raw_df
    .select(
        *[
            F.col(column).cast("string").alias(column)
            for column in source_columns
        ]
    )
    .withColumn("_source_file", F.lit(defects_path))
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("defects"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

write_generation(
    defects_bronze_df,
    "ontologysim.bronze.defects",
)

expected_rows = manifest["source_counts"]["defects"]
actual_rows = (
    spark.table("ontologysim.bronze.defects")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print(f"Expected rows: {expected_rows}")
print(f"Bronze rows:   {actual_rows}")

if expected_rows != actual_rows:
    raise ValueError("Defects ingestion count mismatch")

print("Defects validation passed")

Expected rows: 3216
Bronze rows:   3216
Defects validation passed


In [0]:
measurements_path = (
    f"{landing_base}/measurements/vehicle_signals.parquet"
)

measurements_raw_df = spark.read.parquet(
    measurements_path
)

measurements_raw_df.printSchema()

print(
    "Measurements rows:",
    measurements_raw_df.count(),
)

display(
    measurements_raw_df.limit(10)
)

root
 |-- measurement_id: string (nullable = true)
 |-- timestamp_utc: timestamp (nullable = true)
 |-- execution_ref: string (nullable = true)
 |-- session_ref: string (nullable = true)
 |-- ecu_id: string (nullable = true)
 |-- component_ref: string (nullable = true)
 |-- signal_name: string (nullable = true)
 |-- value_numeric: double (nullable = true)
 |-- value_text: string (nullable = true)
 |-- unit: string (nullable = true)

Measurements rows: 475331


measurement_id,timestamp_utc,execution_ref,session_ref,ecu_id,component_ref,signal_name,value_numeric,value_text,unit
M-RUN-20261001-000001-00001,2026-10-01T07:59:59.123Z,RUN-20261001-000001,SESSION-000001,HCP5,ENERGY_SVC,battery_current_a,0.2,null,A
M-RUN-20261001-000001-00002,2026-10-01T07:59:59.246Z,RUN-20261001-000001,SESSION-000001,HCP5,WAKE_COORD,vehicle_power_state,null,AWAKE,
M-RUN-20261001-000001-00003,2026-10-01T08:00:00.874Z,RUN-20261001-000001,SESSION-000001,ConMod,CELLULAR_MGR,cellular_rssi_dbm,-82.31,null,dBm
M-RUN-20261001-000001-00004,2026-10-01T07:59:59.493Z,RUN-20261001-000001,SESSION-000001,HCP5,ENERGY_SVC,battery_current_a,0.21,null,A
M-RUN-20261001-000001-00005,2026-10-01T07:59:59.616Z,RUN-20261001-000001,SESSION-000001,HCP5,WAKE_COORD,vehicle_power_state,null,AWAKE,
M-RUN-20261001-000001-00006,2026-10-01T08:00:01.249Z,RUN-20261001-000001,SESSION-000001,ConMod,CELLULAR_MGR,cellular_rssi_dbm,-82.61,null,dBm
M-RUN-20261001-000001-00007,2026-10-01T07:59:59.863Z,RUN-20261001-000001,SESSION-000001,HCP5,ENERGY_SVC,battery_current_a,0.21,null,A
M-RUN-20261001-000001-00008,2026-10-01T07:59:59.987Z,RUN-20261001-000001,SESSION-000001,HCP5,WAKE_COORD,vehicle_power_state,null,AWAKE,
M-RUN-20261001-000001-00009,2026-10-01T08:00:01.624Z,RUN-20261001-000001,SESSION-000001,ConMod,CELLULAR_MGR,cellular_rssi_dbm,-82.92,null,dBm
M-RUN-20261001-000001-00010,2026-10-01T08:00:00.233Z,RUN-20261001-000001,SESSION-000001,HCP5,ENERGY_SVC,battery_current_a,0.22,null,A


In [0]:
tm_path = f"{landing_base}/test_management/test_cases.csv"

tm_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(tm_path)
)

expected = manifest["source_counts"]["test_management"]

print("Manifest expected:", expected)
print("Parsed CSV rows:", tm_raw.count())
print("Columns:", tm_raw.columns)

tm_raw.printSchema()

display(tm_raw.limit(10))

Manifest expected: 1005
Parsed CSV rows: 1005
Columns: ['test_case_id', 'title', 'description', 'test_type', 'preconditions', 'test_steps', 'expected_result', 'priority', 'status', 'owner_team', 'version', 'requirement_refs', 'component_refs', 'target_ecu', 'created_at', 'updated_at']
root
 |-- test_case_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- test_type: string (nullable = true)
 |-- preconditions: string (nullable = true)
 |-- test_steps: string (nullable = true)
 |-- expected_result: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- status: string (nullable = true)
 |-- owner_team: string (nullable = true)
 |-- version: string (nullable = true)
 |-- requirement_refs: string (nullable = true)
 |-- component_refs: string (nullable = true)
 |-- target_ecu: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- updated_at: string (nullable = true)



test_case_id,title,description,test_type,preconditions,test_steps,expected_result,priority,status,owner_team,version,requirement_refs,component_refs,target_ecu,created_at,updated_at
TC-BT-000001,Reconnect paired phone after normal vehicle wake,Validate Bluetooth reconnection behavior following a vehicle wake-up transition.,Integration,Vehicle sleeping; paired Bluetooth device available,Wake vehicle; wait for connectivity initialization; observe Bluetooth reconnection,Previously paired Bluetooth device reconnects within the allowed timeout.,High,Approved,ConnectivityValidation,1.0,REQ-HCP3-000054,AuxiliaryComponent0010,HCP3,2026-07-04 08:00:00,2026-07-15 08:00:00
TC-BT-000002,Reconnect Bluetooth after short sleep cycle,Validate Bluetooth reconnection behavior following a vehicle wake-up transition.,System,Vehicle sleeping; paired Bluetooth device available,Wake vehicle; wait for connectivity initialization; observe Bluetooth reconnection,Previously paired Bluetooth device reconnects within the allowed timeout.,High,Approved,ConnectivityValidation,1.0,REQ-SYS-000045,MediaManager,HCP3,2026-07-05 08:00:00,2026-07-17 08:00:00
TC-BT-000003,Reconnect Bluetooth while external connectivity initializes,Validate Bluetooth reconnection behavior following a vehicle wake-up transition.,Integration,Vehicle sleeping; paired Bluetooth device available,Wake vehicle; wait for connectivity initialization; observe Bluetooth reconnection,Previously paired Bluetooth device reconnects within the allowed timeout.,Medium,Approved,ConnectivityValidation,1.0,REQ-SYS-000021,AuxiliaryComponent0069,ConMod,2026-07-06 08:00:00,2026-07-08 08:00:00
TC-BT-000004,Verify Bluetooth retry after delayed wake-state distribution,Validate Bluetooth reconnection behavior following a vehicle wake-up transition.,Regression,Vehicle sleeping; paired Bluetooth device available,Wake vehicle; wait for connectivity initialization; observe Bluetooth reconnection,Previously paired Bluetooth device reconnects within the allowed timeout.,High,Approved,ConnectivityValidation,1.0,REQ-HCP5-000039,PowerStateManager,HCP5,2026-07-07 08:00:00,2026-07-25 08:00:00
TC-BT-000005,Verify reconnect timeout handling,Validate Bluetooth reconnection behavior following a vehicle wake-up transition.,Integration,Vehicle sleeping; paired Bluetooth device available,Wake vehicle; wait for connectivity initialization; observe Bluetooth reconnection,Previously paired Bluetooth device reconnects within the allowed timeout.,Medium,Approved,ConnectivityValidation,1.0,REQ-HCP3-000018,AuxiliaryComponent0052,HCP3,2026-07-08 08:00:00,2026-07-11 08:00:00
TC-BT-000006,Verify Bluetooth state after repeated sleep/wake cycle,Validate Bluetooth reconnection behavior following a vehicle wake-up transition.,System,Vehicle sleeping; paired Bluetooth device available,Wake vehicle; wait for connectivity initialization; observe Bluetooth reconnection,Previously paired Bluetooth device reconnects within the allowed timeout.,High,Approved,ConnectivityValidation,1.0,REQ-HCP3-000030,AuxiliaryComponent0064,HCP3,2026-07-09 08:00:00,2026-07-11 08:00:00
TC-BT-000007,Verify reconnect with degraded connectivity,Validate Bluetooth reconnection behavior following a vehicle wake-up transition.,Functional,Vehicle sleeping; paired Bluetooth device available,Wake vehicle; wait for connectivity initialization; observe Bluetooth reconnection,Previously paired Bluetooth device reconnects within the allowed timeout.,High,Approved,ConnectivityValidation,1.0,REQ-CON-000056,AuxiliaryComponent0009,ConMod,2026-07-10 08:00:00,2026-07-30 08:00:00
TC-BT-000008,Verify paired-device session restoration,Validate Bluetooth reconnection behavior following a vehicle wake-up transition.,Integration,Vehicle sleeping; paired Bluetooth device available,Wake vehicle; wait for connectivity initialization; observe Bluetooth reconnection,Previously paired Bluetooth device reconnects within the allowed timeout.,Medium,Approved,ConnectivityValidation,1.0,REQ-SYS-000093

In [0]:
bronze_v2_tm = (
    spark.table("ontologysim.bronze.test_cases")
    .filter(F.col("_generation_id") == generation_id)
)

print("Bronze V2 rows:", bronze_v2_tm.count())

display(
    bronze_v2_tm
    .groupBy("test_case_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc())
    .limit(20)
)

Bronze V2 rows: 1005


test_case_id,count
TC-CHG-000079,2
TC-BT-000191,2
TC-BT-000139,2
TC-NET-000104,2
TC-NAV-000028,2


In [0]:
print(
    "Missing test_case_id:",
    tm_raw
    .filter(
        F.col("test_case_id").isNull()
        | (F.trim("test_case_id") == "")
    )
    .count()
)

print(
    "Distinct test_case_id:",
    tm_raw
    .select("test_case_id")
    .distinct()
    .count()
)

Missing test_case_id: 0
Distinct test_case_id: 1000


In [0]:
expected_rows = manifest["source_counts"]["test_management"]

bronze_rows = (
    spark.table("ontologysim.bronze.test_cases")
    .filter(F.col("_generation_id") == generation_id)
    .count()
)

print("Expected physical rows:", expected_rows)
print("Bronze physical rows:  ", bronze_rows)

if bronze_rows != expected_rows:
    raise ValueError(
        f"Test Management mismatch: "
        f"expected={expected_rows}, bronze={bronze_rows}"
    )

print("Test Management V2 Bronze validation passed")

Expected physical rows: 1005
Bronze physical rows:   1005
Test Management V2 Bronze validation passed


In [0]:
measurements_path = (
    f"{landing_base}/measurements/vehicle_signals.parquet"
)

measurements_df = (
    spark.read
    .parquet(measurements_path)
    .select(
        "*",
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_generation_id", F.lit(generation_id))
    .withColumn("_profile", F.lit(profile))
    .withColumn("_source_family", F.lit("measurements"))
    .withColumn("_ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

write_generation(
    measurements_df,
    "ontologysim.bronze.measurements",
)

In [0]:
expected_rows = manifest["source_counts"]["measurements"]

bronze_measurements = (
    spark.table("ontologysim.bronze.measurements")
    .filter(F.col("_generation_id") == generation_id)
)

actual_rows = bronze_measurements.count()

print(f"Expected rows: {expected_rows}")
print(f"Bronze rows:   {actual_rows}")

if expected_rows != actual_rows:
    raise ValueError("Measurements row count mismatch")

print("Measurements Bronze validation passed")

Expected rows: 7603360
Bronze rows:   7603360
Measurements Bronze validation passed
